# 🥊 BoxBunny Deployment Notebook

> **Version:** 1.0.0  
> **Last Updated:** February 2026  
> **Package:** boxing_robot_ws

Production deployment notebook for the BoxBunny boxing training system. This notebook provides a complete guide for launching, configuring, and troubleshooting the system.

---

## 📋 Quick Start Checklist

| Step | Action | Status |
|------|--------|--------|
| 1️⃣ | Run **Environment Setup** (Cell 3) | Required |
| 2️⃣ | Run **Build Workspace** (Cell 5) | After code changes only |
| 3️⃣ | Run **Launch System** (Cell 7) | Starts everything! |

---

## ✨ Features

| Feature | Description |
|---------|-------------|
| **Reaction Drill** | Test reflexes with random punch cues using YOLO Pose detection |
| **Shadow Sparring** | Practice punch combos against target sequences |
| **Defence Drill** | Block incoming attacks with motor-controlled targets |
| **AI Coach** | LLM-powered encouragement, trash talk, and performance analysis |

---

## 🎯 Detection Modes

| Mode | Description | Status |
|------|-------------|--------|
| 🎯 **Pose Detection** | YOLO Pose tracks wrist velocity for reaction timing | ✅ Default |
| 🎨 **Color Tracking** | Fast glove colors (Red=Left, Green=Right) for display | ✅ Enabled |
| 🤖 **Action Model** | ML classifier for punch type prediction | ⚠️ Experimental (GPU) |
| 📡 **IMU Sensor** | Glove accelerometer for punch force | ⚠️ Experimental (Hardware) |

---

## 1️⃣ Environment Setup

Creates the shell environment script with all required paths and activations.

**Run this cell first every session** to configure:
- Conda environment (`boxing_ai`)
- ROS 2 Humble workspace
- BoxBunny package overlays
- Library paths for RealSense and CUDA

In [ ]:
%%bash
# =============================================================================
# BoxBunny Environment Setup Script
# Creates a reusable shell script for all subsequent cells
# =============================================================================

cat > /tmp/boxbunny_env.sh <<'SH'
# Activate Conda environment
source ~/miniconda3/etc/profile.d/conda.sh
conda activate boxing_ai

# Prevent user site-packages conflicts
export PYTHONNOUSERSITE=1

# Library paths for RealSense and CUDA
export LD_LIBRARY_PATH=$CONDA_PREFIX/lib:$LD_LIBRARY_PATH

# Fix setuptools compatibility
export SETUPTOOLS_USE_DISTUTILS=stdlib

# Source ROS 2 Humble
source /opt/ros/humble/setup.bash

# Source BoxBunny workspace (if built)
source /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/install/setup.bash 2>/dev/null || true
SH

echo "✅ Environment script created at /tmp/boxbunny_env.sh"

---

## 2️⃣ Build Workspace

Compiles all ROS 2 packages using `colcon build`.

**When to run:**
- After any code changes in `src/` folder
- After pulling updates from version control
- After adding new packages

**Build time:** ~30-60 seconds (incremental builds are faster)

In [ ]:
%%bash
# =============================================================================
# Build BoxBunny Workspace
# Uses symlink-install for faster development iteration
# =============================================================================

source /tmp/boxbunny_env.sh
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws

echo "🔨 Building BoxBunny workspace..."
colcon build --symlink-install

# Source the newly built packages
source install/setup.bash

echo ""
echo "✅ Build complete! Packages ready for launch."

---

## 3️⃣ 🚀 Launch BoxBunny System (Default Mode)

This is the **main deployment command**. Launches the complete boxing training system with Color Tracking mode (default, fast & reliable).

### What Gets Launched

| Component | Description |
|-----------|-------------|
| 📷 **RealSense Camera** | RGB + Depth streaming at 30 FPS |
| 🎨 **Glove Tracker** | Color-based detection (Red=Jab, Green=Cross) |
| 🎯 **Reaction Drill** | Drill state machine with timing |
| 📊 **Punch Analytics** | Statistics aggregation and logging |
| 💬 **AI Coach** | LLM-powered trash talk and encouragement |
| 🖥️ **Main GUI** | Full-screen training interface |

### Expected Startup Time
~5-10 seconds until GUI appears

In [ ]:
%%bash
# =============================================================================
# Launch BoxBunny System (Default Mode)
# Uses Color Tracking for fast, reliable punch detection
# =============================================================================

source /tmp/boxbunny_env.sh
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source install/setup.bash

echo "🚀 Launching BoxBunny Training System..."
echo "   Mode: Color Tracking (default)"
echo "   Press Ctrl+C to stop"
echo ""

# Default deployment: Color Tracking + LLM Coach
ros2 launch boxbunny_bringup boxbunny_deploy.launch.py

# TIP: Run this in a separate terminal to monitor motor commands:
# ros2 topic echo /robot/motor_command

---

## 🧪 Experimental Modes (Optional)

These modes provide enhanced detection capabilities but require additional hardware or GPU resources.

### 4a. Launch with AI Action Model

Uses YOLO Pose + trained ML classifier for more accurate punch type detection.

| Requirement | Details |
|-------------|---------|
| **GPU** | NVIDIA GPU with 4GB+ VRAM |
| **CUDA** | CUDA 11.x or 12.x |
| **Model** | Pre-trained action recognition weights |

In [ ]:
%%bash
source /tmp/boxbunny_env.sh
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source install/setup.bash

# Experimental: Action Model mode (GPU required)
ros2 launch boxbunny_bringup boxbunny_deploy.launch.py detection_mode:=action

### 4b. Launch with IMU Sensor

Adds glove-mounted accelerometer for punch force measurement and precise timing.

| Requirement | Details |
|-------------|---------|
| **Hardware** | MPU6050 IMU sensor on I2C bus |
| **Connection** | I2C address 0x68 (default) |
| **Calibration** | Run `imu_punch_gui` first to calibrate |

In [ ]:
%%bash
source /tmp/boxbunny_env.sh
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source install/setup.bash

# Experimental: With IMU sensor
ros2 launch boxbunny_bringup boxbunny_deploy.launch.py enable_imu:=true

### 4c. Full Experimental Mode (Action Model + IMU)

In [ ]:
%%bash
source /tmp/boxbunny_env.sh
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source install/setup.bash

# Full experimental: Action Model + IMU
ros2 launch boxbunny_bringup boxbunny_deploy.launch.py detection_mode:=action enable_imu:=true

---

## 📖 System Architecture Reference

### ROS 2 Node Overview

| Node | Package | Purpose |
|------|---------|---------|
| `simple_camera_node` | boxbunny_vision | RGB-D camera streaming |
| `glove_tracker_node` | boxbunny_vision | Color-based punch detection |
| `action_predictor` | boxbunny_vision | RGBD action classification |
| `punch_fusion_node` | boxbunny_fusion | Vision + IMU data fusion |
| `reaction_drill_manager` | boxbunny_drills | Drill state machine |
| `shadow_sparring_drill` | boxbunny_drills | Combo training |
| `defence_drill` | boxbunny_drills | Blocking practice |
| `punch_stats_node` | boxbunny_analytics | Statistics aggregation |
| `llm_coach_node` | boxbunny_llm | AI coaching feedback |
| `mpu6050_node` | boxbunny_imu | IMU sensor driver |
| `imu_punch_classifier` | boxbunny_imu | IMU-based classification |

### Key Topics

```bash
# Punch detection events
ros2 topic echo /punch_events

# Drill state updates
ros2 topic echo /drill_state

# AI coach messages
ros2 topic echo /trash_talk

# Glove positions and velocities
ros2 topic echo /glove_detections

# Action model predictions
ros2 topic echo /action_prediction
```

### Key Services

```bash
# Start/stop reaction drill
ros2 service call /start_stop_drill boxbunny_msgs/srv/StartStopDrill "{start: true, num_trials: 5}"

# Generate LLM response
ros2 service call /llm/generate boxbunny_msgs/srv/GenerateLLM "{prompt: 'Give me a tip', mode: 'coach'}"
```

### LLM Coach Modes

| Mode | Description |
|------|-------------|
| `coach` | Technical tips and corrections |
| `encourage` | Positive reinforcement |
| `trash` | Playful competitive taunts |
| `analysis` | Performance breakdown |

---

## 🔧 Troubleshooting & Utilities

### Check Node Status

Verify which ROS 2 nodes are running and what topics are available.

In [ ]:
%%bash
# =============================================================================
# Check ROS 2 Status
# Lists active nodes and topics for debugging
# =============================================================================

source /tmp/boxbunny_env.sh

echo "=== 🤖 Active ROS 2 Nodes ==="
ros2 node list 2>/dev/null || echo "No ROS 2 nodes running"
echo ""

echo "=== 📡 Active Topics ==="
ros2 topic list 2>/dev/null || echo "No topics available"
echo ""

echo "=== 🔌 Active Services ==="
ros2 service list 2>/dev/null | grep -E "drill|llm|calibrate" || echo "No relevant services"

### Kill All BoxBunny Processes

Run this if the system gets stuck or you need a clean restart.

**Warning:** This will terminate all running BoxBunny components.

In [ ]:
%%bash
# =============================================================================
# Emergency Process Cleanup
# Kills all BoxBunny-related processes for a clean restart
# =============================================================================

echo "⚠️  Killing BoxBunny processes..."

# Kill by process name patterns
pkill -f "boxbunny" 2>/dev/null || true
pkill -f "realsense2_camera" 2>/dev/null || true
pkill -f "live_infer_rgbd" 2>/dev/null || true
pkill -f "gui_main" 2>/dev/null || true
pkill -f "simple_camera_node" 2>/dev/null || true

# Wait for processes to terminate
sleep 1

echo "✅ All BoxBunny processes terminated"
echo "   You can now run a fresh launch."

### Test RealSense Camera
Quick check to verify the camera is connected and working.

In [ ]:
%%bash
source /tmp/boxbunny_env.sh
echo "=== RealSense Camera Check ==="
rs-enumerate-devices 2>/dev/null | head -20 || echo "❌ RealSense SDK not found or no camera connected"

### Launch GUI Only (Standalone)
If you need to restart just the GUI without restarting the ROS nodes. **Note:** Uses conda Python directly due to PySide6/Qt compatibility.

In [ ]:
%%bash
source /tmp/boxbunny_env.sh

# GUI must run with conda Python (not ros2 run) due to Qt ABI requirements
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
python src/boxbunny_gui/boxbunny_gui/gui_main.py &
echo "✅ GUI launched"

---

## 📋 Development Reference

### Common Issues & Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| GUI crashes with Qt/xcb error | OpenCV vs PySide6 conflict | Run GUI with `python gui_main.py` directly |
| Camera not detected | USB issue or driver conflict | Run `cleanup_camera.sh`, reconnect USB |
| LLM responses are slow | Model loading or memory | Check GPU memory, reduce batch size |
| Action model not loading | Insufficient VRAM | Use Color Tracking mode instead |
| IMU shows no data | I2C connection issue | Check wiring, verify address with `i2cdetect` |

### Key File Locations

| Component | Path |
|-----------|------|
| **Main GUI** | `src/boxbunny_gui/boxbunny_gui/gui_main.py` |
| **Launch Files** | `src/boxbunny_bringup/launch/` |
| **Vision Nodes** | `src/boxbunny_vision/boxbunny_vision/` |
| **Drill Logic** | `src/boxbunny_drills/boxbunny_drills/` |
| **IMU Nodes** | `src/boxbunny_imu/boxbunny_imu/` |
| **LLM Nodes** | `src/boxbunny_llm/boxbunny_llm/` |
| **YOLO Models** | `models/checkpoints/` |
| **LLM Models** | `models/llm/` |

### Useful Commands

```bash
# Monitor punch events in real-time
ros2 topic echo /punch_events

# Check camera FPS
ros2 topic hz /camera/color/image_raw

# View glove detection debug image
ros2 run rqt_image_view rqt_image_view /glove_debug_image

# Restart just the drill manager
ros2 run boxbunny_drills reaction_drill
```

---

*BoxBunny Training System - Built with ROS 2 Humble, PySide6, and ❤️*